# 12 — GRN Inference and Perturbation Prediction

**Prerequisites:** Run [07_normalization_dimred.ipynb](07_normalization_dimred.ipynb) and [08_perturbation_effects.ipynb](08_perturbation_effects.ipynb) first.
Requires: `data/norman_assigned.h5ad`, `data/de_results.pkl`.

**What you'll learn:**
- Why Perturb-seq data is ideally suited for Gene Regulatory Network (GRN) inference
- Transcription factor (TF) activity inference using `decoupler-py` on DE results
- Correlation-based gene-gene networks from perturbation effect profiles
- GEARS: graph-based deep learning for predicting responses to unseen perturbations
- A critical benchmarking perspective: when GEARS outperforms a linear baseline (and when it doesn't)

**Critical reading:** Kernfeld et al. 2025 (*Nature Methods*) — "Deep-learning-based gene perturbation effect prediction does not yet outperform simple linear baselines"

**Estimated time:** 3–5 hours (GPU-dependent for GEARS)

In [ ]:
import os
import pickle
import scanpy as sc
import pertpy as pt
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings("ignore")

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=100, facecolor="white")

DATA_DIR = "data"
FIG_DIR = "figures"

# Load data
adata = sc.read_h5ad(os.path.join(DATA_DIR, "norman_assigned.h5ad"))

de_path = os.path.join(DATA_DIR, "de_results.pkl")
if os.path.exists(de_path):
    with open(de_path, "rb") as f:
        de_results = pickle.load(f)
    print(f"DE results: {de_results.shape}")
else:
    de_results = None
    print("de_results.pkl not found — run notebook 08 first")

PERT_COL = "perturbation" if "perturbation" in adata.obs.columns else "condition"
print(f"AnnData: {adata.shape}")

## 1. Why Perturb-seq is ideal for GRN inference

Standard single-cell RNA-seq (observational data) can only infer gene co-expression, not causality. If gene A and gene B are correlated across cells, we don't know if A → B, B → A, or they share a common upstream driver.

Perturb-seq provides **causal interventions**: when you knock down transcription factor A and observe a change in gene B's expression, you have direct evidence that A regulates B (or is upstream of B in some regulatory cascade).

This makes Perturb-seq data ideal for constructing **causal GRNs**:

```
Causal GRN edge A → B:
  Perturb gene A → observe change in gene B's expression
  
Compared to correlation-based GRN:
  Observe that genes A and B are correlated across many cells
  (does not distinguish A → B from B → A or A ← C → B)
```

The key limitation: Perturb-seq captures downstream transcriptional cascades, not direct binding events. A gene B responding to perturbation of A may be 3 steps downstream — it's a functional connection, not necessarily a direct binding interaction.

## 2. TF activity inference with decoupler-py

Instead of asking "which TFs are differentially expressed?", we ask "which TFs are *active* — i.e., their target genes are collectively changed?" This is more sensitive and more interpretable.

`decoupler-py` implements this using a weighted sum (ULM, uniform linear model) of DE results against a TF→target gene prior network (CollecTRI or DoRothEA).

In [ ]:
try:
    import decoupler as dc
    
    # Load CollecTRI prior network
    print("Loading CollecTRI TF-target network...")
    collectri = dc.get_collectri(organism="human", split_complexes=False)
    print(f"CollecTRI: {len(collectri)} TF-target interactions, {collectri['source'].nunique()} TFs")
    
    DECOUPLER_AVAILABLE = True
    
except ImportError:
    print("decoupler not available — install with: pip install decoupler-py")
    DECOUPLER_AVAILABLE = False

In [ ]:
if DECOUPLER_AVAILABLE and de_results is not None:
    gene_col = "gene" if "gene" in de_results.columns else de_results.columns[0]
    
    # Build a matrix: perturbations × genes (LFC values)
    lfc_matrix = de_results.pivot_table(
        index="gene_target",
        columns=gene_col,
        values="log2FoldChange",
        aggfunc="mean"
    ).fillna(0)
    
    # Run ULM (Univariate Linear Model) for TF activity
    print(f"Running TF activity inference on {lfc_matrix.shape[0]} perturbations...")
    
    # Create a minimal AnnData for decoupler
    adata_de = anndata.AnnData(
        X=lfc_matrix.values,
        obs=pd.DataFrame(index=lfc_matrix.index),
        var=pd.DataFrame(index=lfc_matrix.columns),
    )
    
    dc.run_ulm(
        mat=adata_de,
        net=collectri,
        source="source",
        target="target",
        weight="weight",
        verbose=False,
    )
    
    # Extract TF activity scores
    tf_activity = adata_de.obsm["ulm_estimate"]
    print(f"TF activity matrix: {tf_activity.shape} (perturbations × TFs)")
    
    # Plot heatmap of top TFs
    tf_activity_df = pd.DataFrame(
        tf_activity.values,
        index=lfc_matrix.index,
        columns=tf_activity.columns if hasattr(tf_activity, "columns") else adata_de.uns.get("ulm_estimate", {}).get("columns", range(tf_activity.shape[1]))
    )
    
    # Select top variable TFs
    top_tfs = tf_activity_df.var(axis=0).nlargest(20).index
    
    g = sns.clustermap(
        tf_activity_df[top_tfs],
        cmap="RdBu_r",
        center=0,
        figsize=(14, 8),
        xticklabels=True,
        yticklabels=True,
        cbar_kws={"label": "TF activity score"},
    )
    g.ax_heatmap.set_title("TF activity across perturbations (CollecTRI + ULM)", pad=10)
    g.savefig(os.path.join(FIG_DIR, "12_tf_activity.png"), dpi=150, bbox_inches="tight")
    plt.show()
    
else:
    print("Skipping TF activity — requires decoupler-py and de_results.pkl")

## 3. Correlation-based GRN from perturbation profiles

A simple causal GRN: compute the correlation between the perturbation effect profiles of different genes. Genes whose knockdown produces similar transcriptional effects are likely in the same pathway or regulatory network.

In [ ]:
# Compute perturbation signatures from the AnnData
ntc_mask = adata.obs["is_ntc"] if "is_ntc" in adata.obs.columns else \
    adata.obs[PERT_COL].str.lower().str.contains("ctrl|control|ntc")

X = adata.X
if scipy.sparse.issparse(X):
    X = X.toarray()

# Normalize if needed
if np.allclose(X[:3, :3], np.round(X[:3, :3])):
    adata_norm = adata.copy()
    sc.pp.normalize_total(adata_norm, target_sum=1e4)
    sc.pp.log1p(adata_norm)
    X = adata_norm.X
    if scipy.sparse.issparse(X):
        X = X.toarray()

ntc_mean = X[ntc_mask.values].mean(axis=0)

# Per-gene mean signatures (single-target cells only)
single_mask = ~ntc_mask & ~(adata.obs.get("is_dual", pd.Series(False, index=adata.obs_names)))
gene_sigs = {}

for gene in adata.obs.loc[single_mask, PERT_COL].unique():
    cell_idx = (adata.obs[PERT_COL] == gene) & single_mask
    mean_expr = X[cell_idx.values].mean(axis=0)
    gene_sigs[gene] = mean_expr - ntc_mean

print(f"Perturbation signatures: {len(gene_sigs)} genes")

In [ ]:
# Build gene × gene correlation matrix of perturbation signatures
gene_list = list(gene_sigs.keys())
sig_matrix = np.array([gene_sigs[g] for g in gene_list])

# Pearson correlation between perturbation profiles
grn_corr = np.corrcoef(sig_matrix)

grn_df = pd.DataFrame(grn_corr, index=gene_list, columns=gene_list)

# Plot top 25 genes by variance in correlation profile
top_grn_genes = grn_df.var(axis=0).nlargest(25).index

g = sns.clustermap(
    grn_df.loc[top_grn_genes, top_grn_genes],
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    figsize=(12, 10),
    xticklabels=True,
    yticklabels=True,
    cbar_kws={"label": "Pearson r (perturbation profiles)"},
)
g.ax_heatmap.set_title("Gene-gene correlation of perturbation effects\n(functional GRN proxy)", pad=10)
g.savefig(os.path.join(FIG_DIR, "12_grn_correlation.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Highly correlated gene pairs act in the same pathway (similar downstream effects).")
print("Anti-correlated pairs may represent opposing regulatory arms.")

## 4. GEARS: deep learning for perturbation prediction

GEARS (Roohani et al. 2023, *Nature Biotechnology*) uses a graph neural network over gene-gene interaction graphs (GO, STRING) to predict transcriptional responses to unseen perturbations.

The key test: can GEARS predict the response to a **held-out single perturbation** that was not seen during training?

**Installation note:** GEARS requires PyTorch and PyTorch Geometric (PyG). These must be installed separately based on your CUDA version.

In [ ]:
# Check if GEARS is available
GEARS_AVAILABLE = False
try:
    import torch
    from gears import PertData, GEARS
    GEARS_AVAILABLE = True
    print(f"GEARS available. PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("No GPU detected — GEARS training will be very slow on CPU")
except ImportError as e:
    print(f"GEARS not available: {e}")
    print()
    print("To install GEARS with PyTorch:")
    print("  1. Install PyTorch: https://pytorch.org/get-started/locally/")
    print("  2. Install PyG: https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html")
    print("  3. pip install cell-gears")
    print()
    print("Falling back to linear ridge regression baseline (section 5).")

In [ ]:
if GEARS_AVAILABLE:
    # Load Norman 2019 in GEARS format
    # GEARS expects raw counts in adata.X and perturbation labels in adata.obs['condition']
    adata_gears = adata.copy()
    
    # Ensure we have raw counts
    if "counts" in adata_gears.layers:
        adata_gears.X = adata_gears.layers["counts"]
    
    # GEARS needs a 'condition' column
    adata_gears.obs["condition"] = adata_gears.obs[PERT_COL].copy()
    
    # GEARS needs 'gene_name' in var
    adata_gears.var["gene_name"] = adata_gears.var_names
    
    print("Preparing GEARS PertData...")
    pert_data = PertData("data/gears_data")
    pert_data.load(data=adata_gears)
    pert_data.prepare_split(split="simulation", seed=42)
    pert_data.get_dataloader(batch_size=32, test_batch_size=128)
    
    print("Train/val/test split:")
    print(f"  Train perturbations: {len(pert_data.split["train"])}")
    print(f"  Val perturbations: {len(pert_data.split['val'])}")
    print(f"  Test perturbations: {len(pert_data.split['test'])}")

In [ ]:
if GEARS_AVAILABLE:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    gears_model = GEARS(pert_data, device=device)
    gears_model.model_initialize(hidden_size=64)
    
    print(f"Training GEARS on {device}...")
    print("(This takes ~30 min on GPU or several hours on CPU)")
    
    gears_model.train(
        epochs=20,
        lr=1e-3,
        weight_decay=1e-5,
    )
    
    # Evaluate on test set
    test_results = gears_model.predict(
        [[p] for p in pert_data.split["test"]]
    )
    print("GEARS evaluation complete.")

## 5. Linear ridge regression baseline

Before concluding that GEARS is useful, we need a baseline. The simplest baseline: predict the transcriptional response to perturbation X by finding the most similar perturbation in the training set (nearest-neighbor) or by a ridge regression on perturbation identity.

**Kernfeld et al. 2025 finding:** On most Perturb-seq datasets, GEARS does not significantly outperform a mean predictor or a linear baseline, particularly for single perturbations on held-out genes. The advantage of GEARS is most pronounced for double perturbations where the interaction structure matters.

In [ ]:
# Linear baseline: predict single perturbation response from gene embedding
# Strategy: represent each gene as its pre-perturbation expression profile,
# use ridge regression to predict the perturbation signature

# Use NTC mean as gene feature (the baseline expression of each target gene)
# This is the simplest possible feature representation

gene_list = list(gene_sigs.keys())
n = len(gene_list)

# Feature: per-gene baseline expression (NTC mean for that gene)
gene_to_var_idx = {g: i for i, g in enumerate(adata.var_names)}
X_features = np.array([
    ntc_mean[gene_to_var_idx[g]] if g in gene_to_var_idx else 0.0
    for g in gene_list
]).reshape(-1, 1)  # shape: (n_genes, 1)

# For a more informative feature, use co-expression of the target gene with all others
# across NTC cells
X_ntc_full = X[ntc_mask.values]
target_gene_expressions = np.array([
    X_ntc_full[:, gene_to_var_idx[g]].mean() if g in gene_to_var_idx else 0.0
    for g in gene_list
]).reshape(-1, 1)

# Target: perturbation signature (summarized as top PCs)
from sklearn.decomposition import PCA
sig_matrix = np.array([gene_sigs[g] for g in gene_list])

pca_sig = PCA(n_components=10, random_state=42)
Y = pca_sig.fit_transform(sig_matrix)  # shape: (n_genes, 10)

print(f"Features: {target_gene_expressions.shape}, Targets: {Y.shape}")

In [ ]:
# Leave-one-out cross-validation for the mean predictor and ridge regression
# Mean predictor: predict each held-out gene's signature as the mean of all other genes' signatures

y_true_all = []
y_pred_mean_all = []
y_pred_ridge_all = []

ridge = Ridge(alpha=1.0)

for i in range(n):
    # Hold out gene i
    train_idx = [j for j in range(n) if j != i]
    
    X_train = target_gene_expressions[train_idx]
    Y_train = Y[train_idx]
    X_test = target_gene_expressions[[i]]
    Y_test = Y[[i]]
    
    # Mean predictor
    mean_pred = Y_train.mean(axis=0, keepdims=True)
    y_pred_mean_all.append(mean_pred.flatten())
    
    # Ridge regression
    if len(np.unique(X_train)) > 1:
        ridge.fit(X_train, Y_train)
        ridge_pred = ridge.predict(X_test)
    else:
        ridge_pred = mean_pred
    y_pred_ridge_all.append(ridge_pred.flatten())
    
    y_true_all.append(Y_test.flatten())

y_true = np.array(y_true_all)
y_pred_mean = np.array(y_pred_mean_all)
y_pred_ridge = np.array(y_pred_ridge_all)

# Per-gene R² (predicting the PC-projected signature)
r2_mean = np.array([r2_score(y_true[i], y_pred_mean[i]) for i in range(n)])
r2_ridge = np.array([r2_score(y_true[i], y_pred_ridge[i]) for i in range(n)])

print(f"Mean predictor — mean R²: {r2_mean.mean():.3f}")
print(f"Ridge baseline  — mean R²: {r2_ridge.mean():.3f}")

In [ ]:
# Compare baselines
methods = ["Mean predictor", "Ridge regression"]
r2_values = [r2_mean, r2_ridge]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plots of R² per method
axes[0].boxplot(r2_values, labels=methods, notch=False, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6))
axes[0].set_ylabel("R² (LOO cross-validation)")
axes[0].set_title("Prediction performance\n(leave-one-out, PC-projected signature)")
axes[0].axhline(0, color="red", linestyle="--", linewidth=1, label="Chance")
axes[0].legend()

# Scatter: mean vs. ridge R²
axes[1].scatter(r2_mean, r2_ridge, alpha=0.5, s=30)
lim = (min(r2_mean.min(), r2_ridge.min()) - 0.05,
       max(r2_mean.max(), r2_ridge.max()) + 0.05)
axes[1].plot(lim, lim, "k--", lw=1)
axes[1].set_xlabel("Mean predictor R²")
axes[1].set_ylabel("Ridge regression R²")
axes[1].set_title("Mean vs. ridge: per-gene R²\n(points above diagonal = ridge better)")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "12_baseline_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Critical perspective: the 2025 benchmarking result

In [ ]:
print("""
=== Kernfeld et al. 2025: Key Findings ===

Paper: "Deep-learning-based gene perturbation effect prediction does not yet
        outperform simple linear baselines" — Nature Methods, 2025.

Key result:
  - GEARS was benchmarked against linear baselines (mean predictor, ridge
    regression) across 14 Perturb-seq datasets
  - On held-out SINGLE perturbations, GEARS does NOT consistently outperform
    a simple mean predictor
  - GEARS shows advantage for DOUBLE perturbations in a few datasets
  - The "virtual cell" hypothesis (learn a generalizable model of perturbation
    response) is not yet supported by current benchmarks

Why does this matter for practitioners?
  - Before using GEARS (or scGPT, PERT, scLAMBDA) to predict perturbation effects,
    always benchmark against the mean predictor
  - A model that barely beats 'predict the mean' is not useful for experimental guidance
  - The task framing matters: predicting differentially expressed genes is harder
    than predicting the mean expression profile

What GEARS IS useful for:
  - Hypothesis generation for double perturbation experiments
  - Settings where you have training data from closely related conditions
  - As part of an ensemble or as a feature extractor, not as a standalone predictor

Recommended workflow:
  1. Define your prediction task precisely (held-out gene? held-out condition?)
  2. Run mean predictor as baseline
  3. If GEARS beats the mean predictor by >5% R², use it
  4. Otherwise, interpret results with caution
""")

## Key takeaways

1. Perturb-seq provides **causal** interventions that are ideal for GRN inference — each perturbation reveals the downstream targets of a gene/TF
2. `decoupler-py` TF activity inference is more sensitive than DE of TF genes — it detects activity changes even when TFs are not themselves differentially expressed
3. Correlation of perturbation effect profiles is a simple, interpretable first-pass GRN approach — pairs with high correlation act in the same pathway
4. GEARS is a powerful architecture but current benchmarks (Kernfeld et al. 2025) show it does not consistently outperform linear baselines on held-out single perturbations
5. **Always benchmark against the mean predictor** before reporting a deep learning model as useful for perturbation prediction

---

## Course complete

You have now worked through the complete Perturb-seq analysis pipeline:

```
Concepts (00–02) → Data (03) → Raw processing (04) → QC (05) → Guide assignment (06)
→ Normalization (07) → DE effects (08) → Escape filtering (09) → Visualization (10)
→ Genetic interactions (11) → GRN + prediction (12)
```

**Where to go next:**
- [pertpy documentation](https://pertpy.readthedocs.io/) — comprehensive single-cell perturbation analysis
- [scverse best practices](https://www.sc-best-practices.org/conditions/perturbation_modeling.html) — chapter on perturbation modeling
- [scPerturb](https://scperturb.org/) — 44 harmonized Perturb-seq datasets for comparative methods development
- [PerturBase](http://www.perturbase.cn/) — 122 datasets with built-in DEG analysis